In [ ]:
import pandas as pd
from datetime import datetime,timedelta
import pandas as pd, numpy as np
import requests
import flexpolyline  # HERE Flexible Polyline 解码
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from math import radians, sin, cos, asin, sqrt
import json
import os 
import sys
import openpyxl
PROJECT_ROOT = os.path.abspath("../..")
sys.path.append(PROJECT_ROOT)
api_keys = json.load(open('../../src/api/api_keys.json'))

In [ ]:
# 导入实际测量的车辆数据，每次实验修改这里就可以！
file_name = "20250412_AY71UCD_Leg1.csv"

In [ ]:
file_path_test = "../../data/processed/AY71UCD"+"/" + file_name
# 读取输入数据
raw_df = pd.read_csv(file_path_test)
origin = raw_df.iloc[0].Latitude, raw_df.iloc[0].Longitude
destination = raw_df.iloc[-1].Latitude, raw_df.iloc[-1].Longitude
raw_df["timestamp"] = pd.to_datetime(raw_df["UnixTime"], unit='ms')
departure_time = raw_df.iloc[0].timestamp
print("origin:", origin)
print("destination:", destination)
print("duration:", pd.to_datetime(raw_df.iloc[-1].UnixTime, unit='ms') - pd.to_datetime(raw_df.iloc[0].UnixTime, unit='ms')) # 转化为时分秒
print("departure_time:", departure_time)
print("distance:", raw_df.iloc[-1].Distance - raw_df.iloc[0].Distance)
print("distance_gps:", raw_df.iloc[-1].distance_gps - raw_df.iloc[0].distance_gps)

In [ ]:
RESULTS_SAVE_DIR = os.path.join('results_2025SRFAnnualMeeting', file_name.split('.')[0])
os.makedirs(RESULTS_SAVE_DIR, exist_ok=True)
request_route_path = os.path.join(RESULTS_SAVE_DIR, 'route.csv')  # 定义请求的routing的结果的路径
predicted_cycle_path = os.path.join(RESULTS_SAVE_DIR, 'predicted_cycle.csv') # 定义预测的drving cycle的结果的路径

#### predict driving cycle

In [ ]:
# 生成via点和passThrough
def generate_via_points_and_pass_through(df, num_points=30):
    '''
    取df的num_points个点作为via点，起点和终点不包括在内
    :param df: 输入的DataFrame，包含经纬度信息
    :param num_points: 需要生成的via点数量
    :return: via_points: 生成的via点列表，pass_through: 是否经过这些via点的布尔列表
    '''
    if num_points < 2:
        raise ValueError("num_points must be at least 2 to include start and end points.")
    via_points = []
    total = len(df)
    # 起点和终点不包括在内，所以i从1到num_points-1
    for i in range(1, num_points):
        idx = int(i * total / (num_points + 1))
        if idx > 0 and idx < total - 1:
            via_points.append((df.iloc[idx].Latitude, df.iloc[idx].Longitude))
    pass_through = [True] * len(via_points)
    return via_points, pass_through

# 生成via和passThrough
via_points, pass_through = generate_via_points_and_pass_through(raw_df, num_points=15)
# 打印生成的via和passThrough
print("via_points:", via_points)
print("pass_through:", pass_through)

In [ ]:
%load_ext autoreload
%autoreload 2 
from src.api.here_api import HereAPIClient
from src.api.google_api import GoogleAPIClient
HereAPIClientInstance = HereAPIClient(api_keys.get('here_api_key')) # 初始化 HERE API
GoogleAPIClientInstance = GoogleAPIClient(api_keys.get('google_api_key')) # 初始化

In [ ]:
if os.path.exists(request_route_path):
    # 如果请求的routing结果已经存在，则直接读取
    route_df = pd.read_csv(request_route_path)
else: 
    # 获取路线响应
    resp = HereAPIClientInstance.get_route_resp(
        origin=origin,
        destination=destination,
        via=via_points,
        passThrough=pass_through,
        departure_time=departure_time,
        transportMode="Truck"
    )
    # 将响应转换为 DataFrame
    route_df = HereAPIClientInstance.resp2df(resp)
    route_df.to_csv(os.path.join(RESULTS_SAVE_DIR, 'route.csv'), index=False)
    print("Route response saved to:", os.path.join(RESULTS_SAVE_DIR, 'route.csv'))

In [ ]:
# TBD: 这里可以添加对route_df的snap to road处理

In [ ]:
# plot trajectory on map to check whether the routing data is same with the recorded driving cycle data
# 这里可以快速检查routing数据和measure数据是否一致!，如果不一致需要手动调整via和passThrough
# 示例：via = [37.7749, -122.4194], [37.7849, -122.4094]], passThrough = [True, True]
import folium
m = folium.Map(location=[raw_df["Latitude"].mean(), raw_df["Longitude"].mean()], zoom_start=9)
# 把 Google map 瓦片作为 TileLayer 叠加
folium.TileLayer(
    tiles=GoogleAPIClientInstance.get_map_tiles(),
    attr='Google',
    name='Google Maps',
    overlay=True,
    control=True,
    max_zoom=20,        # HERE 瓦片最大级别
    min_zoom=1
).add_to(m)
# 画驾驶工况数据轨迹
folium.PolyLine(
    locations=raw_df[["Latitude", "Longitude"]].values.tolist(),
    color="red",
    weight=5,
    popup="Measured Data (GPS)"
).add_to(m)
# 用散点图画routing数据轨迹
folium.PolyLine(
    locations=route_df[["Lat", "Lon"]].values.tolist(),
    color="blue",
    weight=3,
    popup="routing Data (HERE API)"
).add_to(m)
# for idx, row in route_df.iterrows():
#     folium.CircleMarker(
#         location=[row["Lat"], row["Lon"]],
#         radius=2,
#         color="blue",
#         fill=True,
#         fill_color="blue",
#         fill_opacity=0.7,
#         opacity=0.7
#     ).add_to(m)
# 标注起点
start_lat, start_lon = raw_df.iloc[0]["Latitude"], raw_df.iloc[0]["Longitude"]
folium.Marker(
    location=[start_lat, start_lon],
    popup="DEPART",
    icon=folium.Icon(color="green", icon="play")
).add_to(m)
# 标注终点
end_lat, end_lon = raw_df.iloc[-1]["Latitude"], raw_df.iloc[-1]["Longitude"]
folium.Marker(
    location=[end_lat, end_lon],
    popup="ARRIVE",
    icon=folium.Icon(color="red", icon="stop")
).add_to(m)

m.add_child(folium.LatLngPopup())
# 保存map到文件
map_file_path = os.path.join(RESULTS_SAVE_DIR, "measurement&route_on_map.html")
m.save(map_file_path)
print(f"Saving map to {map_file_path}")
m

In [ ]:
route_df

In [ ]:
%load_ext autoreload
%autoreload 2
from src.api.srf_api import SRFAPIClient
SRFAPIClientInsance = SRFAPIClient(api_keys['srf_data_TOKEN'])

In [ ]:
%load_ext autoreload
%autoreload 2 
from src.dcgen.dcgen_v3 import driving_cycle_generator

if not os.path.exists(predicted_cycle_path):
    print(f"Generating predicted driving cycle and saving to {predicted_cycle_path}...")
    
    dcgen = driving_cycle_generator()
    # 创建驾驶工况
    dc_df = dcgen.create_driving_cycle(
        route_df=route_df,
        start_time=departure_time,
        v_cap=25,  # m/s, 25 m/s = 90 km/h, 卡车的最大速度
        v_roundaboutEnter=3.0,  # m/s
        v_turn=4.0,  # m/s
        a_acc=0.58,  # 加速度的默认参考值，m/s² 
        a_dec=0.83,  # 减速度的默认参考值，m/s²
        dt=1.0,  # 仿真时间步长，单位：秒
        v_cruise=24.1,  # m/s, 24.1 m/s = 86.76 km/h, 卡车的巡航速度
        smooth_speed=True # 是否对速度曲线进行平滑处理
    )
    dc_df = SRFAPIClientInsance.get_elevation_from_srf_database(dc_df)
    dc_df.to_csv(predicted_cycle_path, index=False)
    print(f"Predicted driving cycle saved to {predicted_cycle_path}")
else:
    print(f"Loading predicted driving cycle from {predicted_cycle_path}")
    dc_df = pd.read_csv(predicted_cycle_path)
    dc_df['timestamp'] = pd.to_datetime(dc_df['timestamp'])

    

In [ ]:
# 检查route_df的Action列中包含哪些唯一值，以便了解预测的驾驶工况中包含哪些动作类型，是否需要补充到dcgen_v3代码中
print(route_df.Action.unique())

#### 测试生成的完整的driving cycle并生成report

In [ ]:
# 计算平均车辆质量
v_mass = raw_df['MassKg'].dropna().mean()
print("average mass:", v_mass)

# 画车辆质量随距离变化曲线
fig_mass, ax_mass = plt.subplots(figsize=(12, 3))
ax_mass.plot(raw_df['distance_gps']/1000, raw_df['MassKg'], label='Measurement', color='red')
ax_mass.axhline(y=v_mass, color='blue', linestyle='--', label=f'Average Mass: {v_mass:.2f} kg')
ax_mass.set_xlabel('Distance (km)', fontsize=16)
ax_mass.set_ylabel('Mass (kg)', fontsize=16)
ax_mass.set_xlim(0, raw_df['distance_gps'].max()/1000)
ax_mass.set_xticks(np.arange(0, raw_df['distance_gps'].max()/1000 + 10, 50))
ax_mass.tick_params(axis='x', labelsize=14)
ax_mass.tick_params(axis='y', labelsize=14)
ax_mass.grid()
ax_mass.legend(fontsize=14, loc='upper left')
ax_mass.set_title('Vehicle Mass Profile by Distance')
plt.tight_layout()
# fig_mass.savefig(os.path.join(RESULTS_SAVE_DIR, "vehicle_mass_profile.png"), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# 计算fuel consumption
%load_ext autoreload
%autoreload 2 
import src.ecp_models.lvd as lvd

In [ ]:
# 车辆参数
crr = 0.0064 # rolling resistance coefficient
cd = 0.46

In [ ]:
# 计算pred. cycle的功率和燃油消耗率
dc_df['p_wheel'] = dc_df.apply(lambda row: lvd.calculate_wheel_power(
    mass_kg=v_mass,
    gradient_degrees=row['road_gradient'],
    velocity_mps=row['speed'],
    acceleration_mps2=row['acc'],
    rolling_resistance_coeff=crr,
    drag_coefficient=cd,
), axis=1)

dc_df['fuel_rate'] = dc_df.apply(lambda row: lvd.calculate_diesel_consumption_rate(
    wheel_power_watts=row['p_wheel'],
    heating_value_mj_l=36.0,  # 燃料的低位热值 (MJ/L)
    engine_efficiency=0.42,   # 发动机效率
    powertrain_efficiency=0.95,  # 传动效率
    idle_fuel_consumption_l_per_hr=0.0  # 怠速燃油消耗率 (L/h)
), axis=1)

# 计算燃油消耗量 (L)
dc_df['fuel_use_cum'] = (dc_df['fuel_rate'] * (dc_df['timestamp'].diff().dt.total_seconds().fillna(0) / 3600)).cumsum()  # L


In [ ]:
# 通过FuelRate计算fuel_use_measured
raw_df['fuel_use_measured'] = raw_df['FuelRate'] * (raw_df['timestamp'].diff().dt.total_seconds().fillna(0) / 3600)  # L
raw_df['fuel_use_measured'] = raw_df['fuel_use_measured'].cumsum()  # L

# 计算measurement speed profile的功率和燃油消耗率
raw_df['p_wheel'] = raw_df.apply(lambda row: lvd.calculate_wheel_power(
    mass_kg=v_mass,
    gradient_degrees=row['road_gradient'],
    velocity_mps=row['Spd_Kmph_x'] / 3.6,  # 转换为 m/s
    acceleration_mps2=row['Acc_mps2'],
    rolling_resistance_coeff=crr,
    drag_coefficient=cd,
), axis=1)

raw_df['fuel_rate'] = raw_df.apply(lambda row: lvd.calculate_diesel_consumption_rate(
    wheel_power_watts=row['p_wheel'],
    heating_value_mj_l=36.0,  # 燃料的低位热值 (MJ/L)
    engine_efficiency=0.42,   # 发动机效率
    powertrain_efficiency=0.95,  # 传动效率
    idle_fuel_consumption_l_per_hr=0.0  # 怠速燃油消耗率 (L/h)
), axis=1)
# 计算燃油消耗量 (L)
raw_df['fuel_use_cum'] = (raw_df['fuel_rate'] * (raw_df['timestamp'].diff().dt.total_seconds().fillna(0) / 3600)).cumsum()  # L

In [ ]:
# 用表格记录每50km的燃油消耗量以及模型误差

max_distance = max(raw_df['distance_gps'])
distance_intervals = np.arange(0, max_distance, 50000)
if distance_intervals[-1] < max_distance:
    distance_intervals = np.append(distance_intervals, max_distance)

distance_labels = [(min(end / 1000, max_distance / 1000)) for end in distance_intervals[1:]]

fuel_use_stats = pd.DataFrame({
    'Distance (km)': distance_labels,
    'Measurement (L)': [
        raw_df[(raw_df['distance_gps'] >= start) & (raw_df['distance_gps'] < end)]['fuel_use_measured'].iloc[-1]
        if not raw_df[(raw_df['distance_gps'] >= start) & (raw_df['distance_gps'] < end)].empty else np.nan
        for start, end in zip(distance_intervals[:-1], distance_intervals[1:])
    ],
    'Model (L)': [
        raw_df[(raw_df['distance_gps'] >= start) & (raw_df['distance_gps'] < end)]['fuel_use_cum'].iloc[-1]
        if not raw_df[(raw_df['distance_gps'] >= start) & (raw_df['distance_gps'] < end)].empty else np.nan
        for start, end in zip(distance_intervals[:-1], distance_intervals[1:])
    ],
    'Pred. Cycle&Model (L)': [
        dc_df[(dc_df['distance'] >= start) & (dc_df['distance'] < end)]['fuel_use_cum'].iloc[-1]
        if not dc_df[(dc_df['distance'] >= start) & (dc_df['distance'] < end)].empty else np.nan
        for start, end in zip(distance_intervals[:-1], distance_intervals[1:])
    ],
})

fuel_use_stats['Model Error (%)'] = (fuel_use_stats['Measurement (L)'] - fuel_use_stats['Model (L)']) / fuel_use_stats['Measurement (L)'] * 100
fuel_use_stats['Pred. Cycle&Model Error (%)'] = (fuel_use_stats['Measurement (L)'] - fuel_use_stats['Pred. Cycle&Model (L)']) / fuel_use_stats['Measurement (L)'] * 100

fuel_use_stats = fuel_use_stats.round(2)
fuel_use_stats.to_excel(os.path.join(RESULTS_SAVE_DIR, "fuel_use_stats.xlsx"), index=False)
fuel_use_stats

In [ ]:
# 画一个总的report图，包括5个子图，5*1的网格布局
report_fig, report_axes = plt.subplots(5, 1, figsize=(12, 15), sharex=False)

# 添加总标题
report_fig.suptitle('Driving Cycle Report for ' + file_name, fontsize=18, y=1.02)

# 子图1：速度随时间变化
ax1 = report_axes[0]
ax1.plot(raw_df["timestamp"], raw_df["Spd_Kmph_x"], label='Measurement', color='red', linewidth=1)
ax1.plot(dc_df['timestamp'], dc_df['speed']*3.6, label='Predicted', color='#1808F7', linewidth=1)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax1.set_xlabel('Time', fontsize=14)
ax1.set_ylabel('Speed (km/h)', fontsize=14)
ax1.set_ylim(0, 100)
ax1.legend(fontsize=12, loc='best')
ax1.grid()
ax1.tick_params(axis='x', labelsize=12)
ax1.tick_params(axis='y', labelsize=12)
ax1.set_title('Driving Cycle Speed vs Time', fontsize=15)
for label in ax1.get_xticklabels():
    label.set_rotation(45)

# 统一distance相关x轴刻度，范围从0到xmax
xmax = max(raw_df["distance_gps"]) / 1000
xlim = (0, xmax)
if xmax < 50:
    xticks = list(np.arange(0, xmax, 10))
else:
    xticks = list(np.arange(0, xmax, 10))
if not np.isclose(xmax, xticks[-1]):
    xticks.append(round(xmax, 2))

# 子图2：速度随距离变化
ax2 = report_axes[1]
ax2.plot(raw_df["distance_gps"]/1000, raw_df["Spd_Kmph_x"], label='Measurement', color='red', linewidth=1)
ax2.plot(dc_df['distance']/1000, dc_df['speed']*3.6, label='Predicted', color='#1808F7', linewidth=1)
ax2.set_xlabel('Distance (km)', fontsize=14)
ax2.set_ylabel('Speed (km/h)', fontsize=14)
ax2.set_xlim(xlim)
ax2.set_ylim(0, 100)
ax2.set_xticks(xticks)
ax2.legend(fontsize=12, loc='best')
ax2.grid()
ax2.tick_params(axis='x', labelsize=12)
ax2.tick_params(axis='y', labelsize=12)
ax2.set_title('Driving Cycle Speed vs Distance', fontsize=15)
for label in ax2.get_xticklabels():
    label.set_rotation(45)

# 子图3：车辆质量随距离变化
ax3 = report_axes[2]
ax3.plot(raw_df['distance_gps']/1000, raw_df['MassKg'], label='Measurement', color='red')
ax3.axhline(y=v_mass, color='blue', linestyle='--', label=f'Average Mass: {v_mass:.2f} kg')
ax3.set_xlabel('Distance (km)', fontsize=14)
ax3.set_ylabel('Mass (kg)', fontsize=14)
ax3.set_xlim(xlim)
ax3.set_xticks(xticks)
ax3.tick_params(axis='x', labelsize=12)
ax3.tick_params(axis='y', labelsize=12)
ax3.grid()
ax3.legend(fontsize=12, loc='upper left')
ax3.set_title('Vehicle Mass Profile by Distance', fontsize=15)
for label in ax3.get_xticklabels():
    label.set_rotation(45)

# 子图4：燃油消耗量关于距离的对比曲线
ax4 = report_axes[3]
ax4.plot(raw_df['distance_gps']/1000, raw_df['fuel_use_measured'], label='Measurement', color='red', linewidth=2)
ax4.plot(raw_df['distance_gps']/1000, raw_df['fuel_use_cum'], label='Model', color="#03A719", linewidth=2)
ax4.plot(dc_df['distance']/1000, dc_df['fuel_use_cum'], label='Model + Pred. Cycle', color="#1808F7", linewidth=2)
ax4.set_xlabel('Distance (km)', fontsize=14)
ax4.set_ylabel('Fuel Use (L)', fontsize=14)
ax4.set_xlim(xlim)
ax4.set_xticks(xticks)
ax4.legend(fontsize=12, loc='lower right')
ax4.grid()
ax4.tick_params(axis='x', labelsize=12)
ax4.tick_params(axis='y', labelsize=12)
ax4.set_title('Fuel Consumption vs. Distance', fontsize=15)
for label in ax4.get_xticklabels():
    label.set_rotation(45)

# 子图5：每50km燃油消耗统计
ax5 = report_axes[4]
# 在数据最前面增加一个(0, 0)的点
distances = [0.0] + fuel_use_stats["Distance (km)"].tolist()
model_errors = [0.0] + fuel_use_stats["Model Error (%)"].tolist()
pred_errors = [0.0] + fuel_use_stats["Pred. Cycle&Model Error (%)"].tolist()

ax5.plot(distances, model_errors, marker="s", label="Model Error (%)", color="#03A719", linewidth=2)
ax5.plot(distances, pred_errors, marker="^", label="Pred. Cycle&Model Error (%)", color="#1808F7", linewidth=2)

# 标注百分比误差数值，确保标注不会画到图外
ymin, ymax = ax5.get_ylim()
offset_above = 8
offset_below = -15
margin = 2  # 距离y轴上下边界的最小距离

for x, y in zip(distances, model_errors):
    # 限制标注y值在ymin+margin和ymax-margin之间
    if y >= 0:
        y_annot = min(y, ymax - margin)
        xytext = (0, offset_above)
        va = 'bottom'
    else:
        y_annot = max(y, ymin + margin)
        xytext = (0, offset_below)
        va = 'top'
    ax5.annotate(
        f"{y:.1f}%", 
        (x, y_annot), 
        textcoords="offset points", 
        xytext=xytext, 
        ha='center', 
        va=va,
        fontsize=10, 
        color="#03A719",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.5)
    )

for x, y in zip(distances, pred_errors):
    if y >= 0:
        y_annot = min(y, ymax - margin)
        xytext = (0, offset_above)
        va = 'bottom'
    else:
        y_annot = max(y, ymin + margin)
        xytext = (0, offset_below)
        va = 'top'
    ax5.annotate(
        f"{y:.1f}%", 
        (x, y_annot), 
        textcoords="offset points", 
        xytext=xytext, 
        ha='center', 
        va=va,
        fontsize=10, 
        color="#1808F7",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.5)
    )

ax5.tick_params(axis='both', labelsize=12)
ax5.set_xticks(xticks)
ax5.grid(True, linestyle='--', alpha=0.5)
ax5.legend(fontsize=12, loc='best')
ax5.set_xlabel("Distance (km)", fontsize=14)
ax5.set_ylabel("Percentage Error (%)", fontsize=14)
ax5.set_title("Fuel Consumption Percentage Error (per 50km)", fontsize=15)
for label in ax5.get_xticklabels():
    label.set_rotation(45)

plt.tight_layout()
report_fig.savefig(os.path.join(RESULTS_SAVE_DIR, "driving_cycle_report.png"), bbox_inches='tight', dpi=300)
plt.show()

#### 画出指定切片的数据结果并保存

In [ ]:
# 创建一个切片副本以避免修改原始数据，而且也可以自定义切片的数据范围
# 注意：两者的切片范围需要一致
distance_threshold = 150000  # 设置距离阈值(单位：米)，可以根据需要调整
dc_df_slice = dc_df[dc_df['distance'] <= distance_threshold].copy()
raw_df_slice = raw_df[raw_df['distance_gps'] <= distance_threshold].copy()

In [ ]:
# 统一distance相关x轴刻度，范围从0到xmax
xmax = max(raw_df_slice["distance_gps"]) / 1000
xlim = (0, xmax)
if xmax < 50:
    xticks = list(np.arange(0, xmax, 10))
else:
    xticks = list(np.arange(0, xmax, 10))
if not np.isclose(xmax, xticks[-1]):
    xticks.append(round(xmax, 2))
print("xlim:", xlim)
print("xticks:", xticks)

In [ ]:
# 画出驾驶工况的速度曲线对比（时间轴）
fig_time, ax_time = plt.subplots(figsize=(12, 3))
ax_time.plot(raw_df_slice["timestamp"], raw_df_slice["Spd_Kmph_x"], label='Measurement', color='red', linewidth=1)
ax_time.plot(dc_df_slice['timestamp'], dc_df_slice['speed']*3.6, label='Predicted', color='#1808F7', linewidth=1)
ax_time.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax_time.set_xlabel('Time', fontsize=16)
ax_time.set_ylabel('Speed (km/h)', fontsize=16)
ax_time.set_ylim(0, 100)
ax_time.legend(fontsize=14, loc='best')
ax_time.grid()
ax_time.tick_params(axis='x', labelsize=14)
ax_time.tick_params(axis='y', labelsize=14)
plt.tight_layout()
fig_time.savefig(os.path.join(RESULTS_SAVE_DIR, "driving_cycle_speed_vs_time.png"), bbox_inches='tight', dpi=300)
plt.show()

# 画出驾驶工况的速度曲线对比（距离轴）
fig_dist, ax_dist = plt.subplots(figsize=(12, 3))
ax_dist.plot(raw_df_slice["distance_gps"]/1000, raw_df_slice["Spd_Kmph_x"], label='Measured', color='red', linewidth=1)
ax_dist.plot(dc_df_slice['distance']/1000, dc_df_slice['speed']*3.6, label='Predicted', color='#1808F7', linewidth=1)
ax_dist.set_xlabel('Distance (km)', fontsize=16)
ax_dist.set_ylabel('Speed (km/h)', fontsize=16)
ax_dist.set_xlim(xlim)
ax_dist.set_ylim(0, 100)
ax_dist.set_xticks(xticks)
ax_dist.legend(fontsize=14, loc='best')
ax_dist.grid()
ax_dist.tick_params(axis='x', labelsize=14)
ax_dist.tick_params(axis='y', labelsize=14)
plt.tight_layout()
fig_dist.savefig(os.path.join(RESULTS_SAVE_DIR, "driving_cycle_speed_vs_distance_slice.png"), bbox_inches='tight', dpi=300)
plt.show()



In [ ]:
# 画出驾驶工况的速度曲线,测试指定距离区间，并将所有子图画在一副图上
print(f"Available distance range: {dc_df_slice['distance'].min()} to {dc_df_slice['distance'].max()} meters")
distance_interval = 10000  # 10 km
num_intervals = int(np.ceil((dc_df_slice['distance'].max() - dc_df_slice['distance'].min()) / distance_interval))

fig, axes = plt.subplots(num_intervals, 1, figsize=(12, 2 * num_intervals), sharex=False)

if num_intervals == 1:
    axes = [axes]

for i in range(num_intervals):
    current_start_distance = dc_df_slice['distance'].min() + i * distance_interval
    current_end_distance = current_start_distance + distance_interval

    dc_df_filtered = dc_df_slice[(dc_df_slice['distance'] >= current_start_distance) & (dc_df_slice['distance'] < current_end_distance)]
    raw_df_filtered = raw_df[(raw_df['distance_gps'] >= current_start_distance) & (raw_df['distance_gps'] < current_end_distance)]

    ax = axes[i]
    ax.plot(dc_df_filtered['distance'], dc_df_filtered['speed'] * 3.6, label='Artificial Speed (km/h)', color='#1808F7')
    ax.plot(raw_df_filtered['distance_gps'], raw_df_filtered['Spd_Kmph_x'], label='Recorded Speed (km/h)', color='red')
    ax.set_xlabel('Distance (m)', fontsize=14)
    ax.set_ylabel('Speed (km/h)', fontsize=14)
    ax.set_xlim(current_start_distance, current_end_distance)
    ax.set_ylim([0, 105])
    ax.grid()
    ax.set_title(f'{current_start_distance/1000:.1f} - {current_end_distance/1000:.1f} km', fontsize=12)
    if i == 0:
        ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_SAVE_DIR, "driving_cycle_speed_vs_distance_intervals_slice.png"), bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
# 画出燃油消耗量关于距离的对比曲线（优化版）
fig_fuel, ax_fuel = plt.subplots(figsize=(12, 3))

ax_fuel.plot(raw_df_slice['distance_gps']/1000, raw_df_slice['fuel_use_measured'], label='Measured', color='red', linewidth=2)
ax_fuel.plot(raw_df_slice['distance_gps']/1000, raw_df_slice['fuel_use_cum'], label='Physics-based Model + Measured Speed', color="#03A719", linewidth=2)
ax_fuel.plot(dc_df_slice['distance']/1000, dc_df_slice['fuel_use_cum'], label='Physics-based Model + Predicted Speed', color="#1808F7", linewidth=2)

ax_fuel.set_xlabel('Distance (km)', fontsize=16)
ax_fuel.set_ylabel('Fuel Use (L)', fontsize=16)
ax_fuel.set_xlim(xlim)
ax_fuel.set_xticks(xticks)
ax_fuel.legend(fontsize=14, loc='lower right')
ax_fuel.grid()
ax_fuel.tick_params(axis='x', labelsize=14)
ax_fuel.tick_params(axis='y', labelsize=14)
plt.tight_layout()
fig_fuel.savefig(os.path.join(RESULTS_SAVE_DIR, "driving_cycle_fuel_use_vs_distance_slice.png"), bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
# 用表格记录切片数据的每50km的燃油消耗量以及模型误差
max_distance = max(raw_df_slice['distance_gps'])
distance_intervals = np.arange(0, max_distance, 50000)
if distance_intervals[-1] < max_distance:
    distance_intervals = np.append(distance_intervals, max_distance)

distance_labels = [(min(end / 1000, max_distance / 1000)) for end in distance_intervals[1:]]

fuel_use_stats_slice = pd.DataFrame({
    'Distance (km)': distance_labels,
    'Measurement (L)': [
        raw_df_slice[(raw_df_slice['distance_gps'] >= start) & (raw_df_slice['distance_gps'] < end)]['fuel_use_measured'].iloc[-1]
        if not raw_df_slice[(raw_df_slice['distance_gps'] >= start) & (raw_df_slice['distance_gps'] < end)].empty else np.nan
        for start, end in zip(distance_intervals[:-1], distance_intervals[1:])
    ],
    'Model (L)': [
        raw_df_slice[(raw_df_slice['distance_gps'] >= start) & (raw_df_slice['distance_gps'] < end)]['fuel_use_cum'].iloc[-1]
        if not raw_df_slice[(raw_df_slice['distance_gps'] >= start) & (raw_df_slice['distance_gps'] < end)].empty else np.nan
        for start, end in zip(distance_intervals[:-1], distance_intervals[1:])
    ],
    'Pred. Cycle&Model (L)': [
        dc_df_slice[(dc_df_slice['distance'] >= start) & (dc_df_slice['distance'] < end)]['fuel_use_cum'].iloc[-1]
        if not dc_df_slice[(dc_df_slice['distance'] >= start) & (dc_df_slice['distance'] < end)].empty else np.nan
        for start, end in zip(distance_intervals[:-1], distance_intervals[1:])
    ],
})

fuel_use_stats_slice['Model Error (%)'] = (fuel_use_stats_slice['Measurement (L)'] - fuel_use_stats_slice['Model (L)']) / fuel_use_stats_slice['Measurement (L)'] * 100
fuel_use_stats_slice['Pred. Cycle&Model Error (%)'] = (fuel_use_stats_slice['Measurement (L)'] - fuel_use_stats_slice['Pred. Cycle&Model (L)']) / fuel_use_stats_slice['Measurement (L)'] * 100

fuel_use_stats_slice = fuel_use_stats_slice.round(2)
fuel_use_stats_slice.to_excel(os.path.join(RESULTS_SAVE_DIR, "fuel_use_stats_slice.xlsx"), index=False)
fuel_use_stats_slice


In [ ]:
plt.figure(figsize=(12, 3))
ax = plt.gca()

# 折线图：Y为百分比误差
ax.plot(fuel_use_stats_slice["Distance (km)"], fuel_use_stats_slice["Model Error (%)"], marker="s", label="Model Error (%)", color="#03A719", linewidth=2)
ax.plot(fuel_use_stats_slice["Distance (km)"], fuel_use_stats_slice["Pred. Cycle&Model Error (%)"], marker="^", label="Pred. Cycle&Model Error (%)", color="#1808F7", linewidth=2)

# 标注百分比误差数值
ymin, ymax = ax.get_ylim()
offset_above = 8
offset_below = -15
margin = 2

for x, y in zip(fuel_use_stats_slice["Distance (km)"], fuel_use_stats_slice["Model Error (%)"]):
        if y >= 0:
                y_annot = min(y, ymax - margin)
                xytext = (0, offset_above)
                va = 'bottom'
        else:
                y_annot = max(y, ymin + margin)
                xytext = (0, offset_below)
                va = 'top'
        ax.annotate(
                f"{y:.1f}%", 
                (x, y_annot), 
                textcoords="offset points", 
                xytext=xytext, 
                ha='center', 
                va=va,
                fontsize=10, 
                color="#03A719",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7)
        )

for x, y in zip(fuel_use_stats_slice["Distance (km)"], fuel_use_stats_slice["Pred. Cycle&Model Error (%)"]):
        if y >= 0:
                y_annot = min(y, ymax - margin)
                xytext = (0, offset_above)
                va = 'bottom'
        else:
                y_annot = max(y, ymin + margin)
                xytext = (0, offset_below)
                va = 'top'
        ax.annotate(
                f"{y:.1f}%", 
                (x, y_annot), 
                textcoords="offset points", 
                xytext=xytext, 
                ha='center', 
                va=va,
                fontsize=10, 
                color="#1808F7",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7)
        )

ax.tick_params(axis='both', labelsize=14)
ax.set_xlim(xlim)
ax.set_xticks(xticks)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(fontsize=14, loc='best')
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Percentage Error (%)")
ax.set_title("Fuel Consumption Percentage Error vs. Distance (Slice)")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_SAVE_DIR, "fuel_use_stats_slice_pct_error.png"), dpi=300)
plt.show()


In [ ]:
# 画一个总的report图，包括5个子图，5*1的网格布局（slice数据）
report_fig, report_axes = plt.subplots(5, 1, figsize=(12, 15), sharex=False)

# 添加总标题
report_fig.suptitle('Driving Cycle Report (Slice) for ' + file_name, fontsize=18, y=1.02)

# 子图1：速度随时间变化
ax1 = report_axes[0]
ax1.plot(raw_df_slice["timestamp"], raw_df_slice["Spd_Kmph_x"], label='Measurement', color='red', linewidth=1)
ax1.plot(dc_df_slice['timestamp'], dc_df_slice['speed']*3.6, label='Predicted', color='#1808F7', linewidth=1)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax1.set_xlabel('Time', fontsize=14)
ax1.set_ylabel('Speed (km/h)', fontsize=14)
ax1.set_ylim(0, 100)
ax1.legend(fontsize=12, loc='best')
ax1.grid()
ax1.tick_params(axis='x', labelsize=12)
ax1.tick_params(axis='y', labelsize=12)
ax1.set_title('Driving Cycle Speed vs Time', fontsize=15)

# 子图2：速度随距离变化
ax2 = report_axes[1]
ax2.plot(raw_df_slice["distance_gps"]/1000, raw_df_slice["Spd_Kmph_x"], label='Measurement', color='red', linewidth=1)
ax2.plot(dc_df_slice['distance']/1000, dc_df_slice['speed']*3.6, label='Predicted', color='#1808F7', linewidth=1)
ax2.set_xlabel('Distance (km)', fontsize=14)
ax2.set_ylabel('Speed (km/h)', fontsize=14)
ax2.set_xlim(0, max(raw_df_slice["distance_gps"])/1000 + 10)
ax2.set_ylim(0, 100)
ax2.set_xticks(np.arange(0, max(raw_df_slice["distance_gps"])/1000 + 10, 10))
ax2.legend(fontsize=12, loc='best')
ax2.grid()
ax2.tick_params(axis='x', labelsize=12)
ax2.tick_params(axis='y', labelsize=12)
ax2.set_title('Driving Cycle Speed vs Distance', fontsize=15)

# 子图3：车辆质量随距离变化
ax3 = report_axes[2]
ax3.plot(raw_df_slice['distance_gps']/1000, raw_df_slice['MassKg'], label='Measurement', color='red')
ax3.axhline(y=v_mass, color='blue', linestyle='--', label=f'Average Mass: {v_mass:.2f} kg')
ax3.set_xlabel('Distance (km)', fontsize=14)
ax3.set_ylabel('Mass (kg)', fontsize=14)
ax3.set_xlim(0, raw_df_slice['distance_gps'].max()/1000)
ax3.set_xticks(np.arange(0, raw_df_slice['distance_gps'].max()/1000 + 10, 10))
ax3.tick_params(axis='x', labelsize=12)
ax3.tick_params(axis='y', labelsize=12)
ax3.grid()
ax3.legend(fontsize=12, loc='upper left')
ax3.set_title('Vehicle Mass Profile by Distance', fontsize=15)

# 子图4：燃油消耗量关于距离的对比曲线
ax4 = report_axes[3]
ax4.plot(raw_df_slice['distance_gps']/1000, raw_df_slice['fuel_use_measured'], label='Measurement', color='red', linewidth=2)
ax4.plot(raw_df_slice['distance_gps']/1000, raw_df_slice['fuel_use_cum'], label='Model', color="#03A719", linewidth=2)
ax4.plot(dc_df_slice['distance']/1000, dc_df_slice['fuel_use_cum'], label='Model + Pred. Cycle', color="#1808F7", linewidth=2)
ax4.set_xlabel('Distance (km)', fontsize=14)
ax4.set_ylabel('Fuel Use (L)', fontsize=14)
ax4.set_xlim(0, max(raw_df_slice['distance_gps'])/1000 + 10)
ax4.set_xticks(np.arange(0, max(raw_df_slice['distance_gps'])/1000 + 10, 10))
ax4.legend(fontsize=12, loc='lower right')
ax4.grid()
ax4.tick_params(axis='x', labelsize=12)
ax4.tick_params(axis='y', labelsize=12)
ax4.set_title('Fuel Consumption vs. Distance', fontsize=15)

# 子图5：每50km燃油消耗统计（slice数据）
ax5 = report_axes[4]
# 在数据最前面增加一个(0, 0)的点
distances = [0.0] + fuel_use_stats_slice["Distance (km)"].tolist()
model_errors = [0.0] + fuel_use_stats_slice["Model Error (%)"].tolist()
pred_errors = [0.0] + fuel_use_stats_slice["Pred. Cycle&Model Error (%)"].tolist()

ax5.plot(distances, model_errors, marker="s", label="Model Error (%)", color="#03A719", linewidth=2)
ax5.plot(distances, pred_errors, marker="^", label="Pred. Cycle&Model Error (%)", color="#1808F7", linewidth=2)

# 标注百分比误差数值，确保标注不会画到图外
ymin, ymax = ax5.get_ylim()
offset_above = 8
offset_below = -15
margin = 2  # 距离y轴上下边界的最小距离

for x, y in zip(distances, model_errors):
    # 限制标注y值在ymin+margin和ymax-margin之间
    if y >= 0:
        y_annot = min(y, ymax - margin)
        xytext = (0, offset_above)
        va = 'bottom'
    else:
        y_annot = max(y, ymin + margin)
        xytext = (0, offset_below)
        va = 'top'
    ax5.annotate(
        f"{y:.1f}%", 
        (x, y_annot), 
        textcoords="offset points", 
        xytext=xytext, 
        ha='center', 
        va=va,
        fontsize=10, 
        color="#03A719",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.5)
    )

for x, y in zip(distances, pred_errors):
    if y >= 0:
        y_annot = min(y, ymax - margin)
        xytext = (0, offset_above)
        va = 'bottom'
    else:
        y_annot = max(y, ymin + margin)
        xytext = (0, offset_below)
        va = 'top'
    ax5.annotate(
        f"{y:.1f}%", 
        (x, y_annot), 
        textcoords="offset points", 
        xytext=xytext, 
        ha='center', 
        va=va,
        fontsize=10, 
        color="#1808F7",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.5)
    )

ax5.tick_params(axis='both', labelsize=12)
ax5.set_xticks(xticks)
ax5.grid(True, linestyle='--', alpha=0.5)
ax5.legend(fontsize=12, loc='best')
ax5.set_xlabel("Distance (km)", fontsize=14)
ax5.set_ylabel("Percentage Error (%)", fontsize=14)
ax5.set_title("Fuel Consumption Percentage Error (per 50km, slice)", fontsize=15)

plt.tight_layout()
report_fig.savefig(os.path.join(RESULTS_SAVE_DIR, "driving_cycle_report_slice.png"), bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
# 保存切片后的地图
import folium
m_sliced = folium.Map(location=[raw_df_slice["Latitude"].mean(), raw_df_slice["Longitude"].mean()], zoom_start=9)
# 把 Google map 瓦片作为 TileLayer 叠加
folium.TileLayer(
    tiles=GoogleAPIClientInstance.get_map_tiles(),
    attr='Google',
    name='Google Maps',
    overlay=True,
    control=True,
    max_zoom=20,        # HERE 瓦片最大级别
    min_zoom=1
).add_to(m_sliced)
# 画routing数据轨迹
folium.PolyLine(
    locations=raw_df_slice[["Latitude", "Longitude"]].values.tolist(),
    color="red",
    weight=5,
    popup="Measured Data (GPS)"
).add_to(m_sliced)
# 画驾驶工况数据轨迹
folium.PolyLine(
    locations=dc_df_slice[["Lat", "Lon"]].values.tolist(),
    color="blue",
    weight=3,
    popup="dc Data (Artificial Driving Cycle)"
).add_to(m_sliced)
# 标注起点（切片数据）
start_lat, start_lon = raw_df_slice.iloc[0]["Latitude"], raw_df_slice.iloc[0]["Longitude"]
folium.Marker(
    location=[start_lat, start_lon],
    popup="DEPART",
    icon=folium.Icon(color="green", icon="play")
).add_to(m_sliced)
# 标注终点（切片数据）
end_lat, end_lon = raw_df_slice.iloc[-1]["Latitude"], raw_df_slice.iloc[-1]["Longitude"]
folium.Marker(
    location=[end_lat, end_lon],
    popup="ARRIVE",
    icon=folium.Icon(color="red", icon="stop")
).add_to(m_sliced)
m_sliced.add_child(folium.LatLngPopup())
# 保存map到文件
map_file_path = os.path.join(RESULTS_SAVE_DIR, "measurement&dc_on_map_slice.html")
m_sliced.save(map_file_path)
print(f"Saving map to {map_file_path}")
m_sliced